In [ ]:
!pip install numpy scipy rustworkx qiskit

In [ ]:
import numpy as np
import rustworkx as rx                                   # graphs
from scipy.optimize import minimize                      # the classical optimizers

from qiskit.quantum_info import SparsePauliOp, Statevector   # cost function + simulator
from qiskit.circuit.library import QAOAAnsatz                # ready-made QAOA circuit

# Step 1: the Max-Cut problem (helpers and brute-force answer)
def make_graph(n, seed=0):
    """Make a random connected graph with n nodes (about half of all possible edges)."""
    for s in range(seed, seed + 1000):
        graph = rx.undirected_gnp_random_graph(n, 0.5, seed=s)
        if rx.is_connected(graph) and graph.num_edges() >= n:
            return graph

def cut_value(bits, graph):
    """Count the edges whose two ends have different colors (different bits)."""
    return sum(bits[u] != bits[v] for u, v in graph.edge_list())

def brute_force_max_cut(graph):
    """Try every possible coloring and return the best cut and one coloring that achieves it."""
    n = graph.num_nodes()
    best_cut, best_bits = -1, None
    for i in range(2 ** n):
        bits = [(i >> k) & 1 for k in range(n)]
        c = cut_value(bits, graph)
        if c > best_cut:
            best_cut, best_bits = c, bits
    return best_cut, best_bits

# Step 2: cost function and QAOA circuit
def cost_hamiltonian(graph):
    """The cost function: one Z_i Z_j term for every edge."""
    n = graph.num_nodes()
    terms = [("ZZ", [u, v], 1.0) for u, v in graph.edge_list()]
    return SparsePauliOp.from_sparse_list(terms, n)

def build_circuit(graph, p):
    """The QAOA circuit with p layers. It has 2*p knobs (p betas and p gammas)."""
    return QAOAAnsatz(cost_operator=cost_hamiltonian(graph), reps=p)

# Step 3: run QAOA (optimizer loop) and helper for the 'chance of perfect answer'
def run_qaoa(graph, p, method="COBYLA", seed=0, maxiter=300):
    """Run QAOA on a graph and return the results in a dictionary."""
    n, num_edges = graph.num_nodes(), graph.num_edges()
    cost = cost_hamiltonian(graph)

    # Build the circuit once and break it into basic gates (this makes every run much faster)
    circuit = build_circuit(graph, p).decompose(reps=5)

    history = []                                         # the cost after each optimizer step

    def average_cost(knobs):
        state = Statevector(circuit.assign_parameters(knobs))     # run the circuit (exactly)
        value = state.expectation_value(cost).real                # average cost
        history.append(value)
        return value

    # 1. random starting values for the knobs
    rng = np.random.default_rng(seed)
    start = rng.uniform(0, np.pi, size=circuit.num_parameters)

    # 2-4. let the classical optimizer tune the knobs
    result = minimize(average_cost, start, method=method, options={"maxiter": maxiter})

    # Probability of every possible coloring with the tuned knobs
    final_state = Statevector(circuit.assign_parameters(result.x))
    probs = final_state.probabilities_dict()             # e.g. {"01101": 0.12, ...}

    most_likely = max(probs, key=probs.get)
    most_likely_bits = [int(c) for c in most_likely[::-1]]     # Qiskit writes qubit 0 on the right

    return {
        "knobs": result.x,
        "average_cut": (num_edges - result.fun) / 2,
        "history": history,
        "steps": len(history),
        "probs": probs,
        "most_likely_bits": most_likely_bits,
    }

def chance_of_perfect(probs, graph, best_cut):
    """Total probability of the colorings that achieve the best possible cut."""
    return sum(pr for key, pr in probs.items()
               if cut_value([int(c) for c in key[::-1]], graph) == best_cut)

# First run on the 5-node example graph
def first_run():
    g5 = make_graph(5, seed=3)
    best5, bits5 = brute_force_max_cut(g5)
    print(f"Example graph: {g5.num_nodes()} nodes, {g5.num_edges()} edges.")
    print(f"Brute force says the best possible cut is {best5} edges.")

    # A tiny example circuit: a 4-node ring, one layer
    ring = rx.PyGraph()
    ring.add_nodes_from(range(4))
    ring.add_edges_from([(0, 1, 1.0), (1, 2, 1.0), (2, 3, 1.0), (3, 0, 1.0)])
    example_circuit = build_circuit(ring, p=1)
    print("Knobs the optimizer can turn:", [str(p) for p in example_circuit.parameters])

    result = run_qaoa(g5, p=2, method="COBYLA", seed=1)
    quality = result["average_cut"] / best5
    perfect = chance_of_perfect(result["probs"], g5, best5)
    most_likely_cut = cut_value(result["most_likely_bits"], g5)
    print(f"Average cut found by QAOA : {result['average_cut']:.2f}")
    print(f"Best possible cut         : {best5}")
    print(f"Quality score             : {quality:.2f}   (1.00 would be perfect)")
    print(f"Chance of a perfect answer: {perfect:.0%}")
    print(f"Optimizer steps used      : {result['steps']}")
    print(f"Most likely coloring      : {result['most_likely_bits']} (cut = {most_likely_cut})")

# Experiment 1: compare optimizers
def experiment_optimizers():
    g8 = make_graph(8, seed=11)
    best8, _ = brute_force_max_cut(g8)
    print(f"\n8-node graph: {g8.num_edges()} edges, best possible cut = {best8}\n")

    optimizers = {"COBYLA": "gradient-free", "Powell": "gradient-free",
                  "BFGS": "gradient-based", "L-BFGS-B": "gradient-based"}
    n_starts = 6

    quality = {}      # quality scores for every start
    for name in optimizers:
        runs = [run_qaoa(g8, p=2, method=name, seed=100 + s) for s in range(n_starts)]
        quality[name] = [r["average_cut"] / best8 for r in runs]
        steps = [r["steps"] for r in runs]
        print(f"{name:9s} ({optimizers[name]:14s}) average quality {np.mean(quality[name]):.3f}"
              f"   | steps used {np.mean(steps):.0f}")
    return optimizers, quality

# Experiment 2: effect of graph size (slowest part)
def experiment_size():
    sizes = [4, 5, 6, 7, 8, 9]
    graphs_per_size, starts = 4, 3

    size_quality = []
    for n in sizes:
        qualities, perfects = [], []
        for gi in range(graphs_per_size):
            graph = make_graph(n, seed=1000 * n + gi)
            best_cut, _ = brute_force_max_cut(graph)
            runs = [run_qaoa(graph, p=2, method="COBYLA", seed=s) for s in range(starts)]
            top_run = max(runs, key=lambda r: r["average_cut"])         # keep the best of the starts
            qualities.append(top_run["average_cut"] / best_cut)
            perfects.append(chance_of_perfect(top_run["probs"], graph, best_cut))
        size_quality.append(np.mean(qualities))
        print(f"{n} nodes: quality {size_quality[-1]:.3f}   chance of perfect answer {np.mean(perfects):.0%}")
    return sizes, size_quality

# Experiment 3: effect of circuit depth
def experiment_depth():
    g8 = make_graph(8, seed=11)
    best8, _ = brute_force_max_cut(g8)
    depths = [1, 2, 3, 4]
    n_starts = 6

    depth_quality = []
    for p in depths:
        runs = [run_qaoa(g8, p=p, method="COBYLA", seed=200 + s, maxiter=500) for s in range(n_starts)]
        q = [r["average_cut"] / best8 for r in runs]
        top_run = max(runs, key=lambda r: r["average_cut"])
        depth_quality.append(np.mean(q))
        perfect = chance_of_perfect(top_run["probs"], g8, best8)
        print(f"p = {p}: quality {depth_quality[-1]:.3f}   chance of perfect answer (best run) {perfect:.0%}")
    return depths, depth_quality

# Main function to run everything and print the summary table
def main():
    first_run()
    optimizers, exp1_quality = experiment_optimizers()
    print()
    sizes, size_quality = experiment_size()
    print()
    depths, depth_quality = experiment_depth()

    # Summary table of all results
    print(f"\n{'Experiment':<30}{'Setting':<18}{'Quality score':>14}")
    print("-" * 62)
    for name in optimizers:
        print(f"{'1: optimizer (8 nodes, p=2)':<30}{name:<18}{np.mean(exp1_quality[name]):>14.3f}")
    for n, v in zip(sizes, size_quality):
        print(f"{'2: size (p=2, COBYLA)':<30}{str(n) + ' nodes':<18}{v:>14.3f}")
    for p, v in zip(depths, depth_quality):
        print(f"{'3: depth (8 nodes, COBYLA)':<30}{'p = ' + str(p):<18}{v:>14.3f}")

if __name__ == "__main__":
    main()